In [1]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from torch.utils.data import Subset


In [2]:
#Process image
transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])


dataset = datasets.ImageFolder('train', transform=transforms)

#80% for training, 20% for validation
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

#Split dataset into training and validation sets
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
train_dataset = Subset(train_dataset, range(500))
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)

In [3]:
import torchvision.models as models
import torch.nn as nn

#Load pre-trained ResNet-18 model, this model works by constantly adding original input to convolutional layers
model = models.resnet18(pretrained=True)    
model.fc = nn.Linear(model.fc.in_features, 10)  # 10 classes of creatures

c:\Users\minhp\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\minhp\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [4]:
#Feature extraction: freeze base model and train only new added classification layer

#Goes through all layers says dont edit except for the last layer 
for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

In [5]:
import torch.optim as optim

#trains the model for 1 epoch, calculates loss, and updates the weights of the last layer using Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

for epoch in range(1):  # Train for 1
    model.train()
    for images, labels in train_loader:
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()



In [6]:

import torch

#Evaluate the model on the validation set, calculate accuracy, and print the result
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        outputs = model(images)
        _, predicted = outputs.max(1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("accuracy", correct / total)

accuracy 0.2744807121661721


In [ ]:
#Unfreeze top layers and fine tune

for param in model.parameters():
    param.requires_grad = False

for param in model.layer4.parameters():
    param.requires_grad = True

for param in model.fc.parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.parameters(), lr=0.0001)

for epoch in range(1):  # Train for 1
    model.train()
    for images, labels in train_loader:
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


#Evaluate the model on the validation set, calculate accuracy, and print the result
model.eval()
correct = 0
total = 0

#This loop goes through the validation set, makes predictions, and compares them to the true labels to calculate accuracy
#and prints the final accuracy after evaluating the model on the validation set
with torch.no_grad():
    for images, labels in val_loader:
        outputs = model(images)
        _, predicted = outputs.max(1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("accuracy", correct / total)


accuracy 0.5326409495548962
